# LDCC 다국어 데이터셋 탐색

LDCC(Language Data Curation Center)의 일본어/중국어 사전학습 데이터셋을 탐색합니다.

**데이터셋:**
- `LDCC/Final_Verification_Data_Labeling_Data_03.Multilingual_Pre-Learning_Data_01.Japanese`
- `LDCC/Final_Verification_Data_Labeling_Data_03.Multilingual_Pre-Learning_Data_02.Chinese`

**결론:**
- 일본어: 1,605,024 samples (~1,815 tokens/sample)
- 중국어: 899,327 samples (~2,535 tokens/sample)
- WebDataset 형식 오류로 `load_dataset()` 직접 사용 불가
- TAR 파일에서 JSON을 직접 읽어야 함

## 1. 환경 설정

In [ ]:
import tarfile
import json
from huggingface_hub import hf_hub_download, HfFolder

# HuggingFace 로그인 확인
token = HfFolder.get_token()
if token:
    print("✅ HuggingFace token found")
else:
    print("⚠️  No token. Run: huggingface-cli login")

## 2. 일본어 데이터셋 탐색

In [ ]:
# 일본어 데이터셋
ja_dataset_name = "LDCC/Final_Verification_Data_Labeling_Data_03.Multilingual_Pre-Learning_Data_01.Japanese"

print("[Japanese Dataset]")
print("Downloading TAR file...")

# TAR 파일 다운로드
tar_path = hf_hub_download(
    repo_id=ja_dataset_name,
    filename="Final_Verification_Data_Labeling_Data_03.Multilingual_Pre-Learning_Data_01.Japanese.tar.gz",
    repo_type="dataset"
)
print(f"✅ Downloaded: {tar_path}")

# TAR 구조 확인
with tarfile.open(tar_path, 'r:gz') as tar:
    all_members = tar.getmembers()
    json_files = [m for m in all_members if m.name.endswith('.json')]
    
    print(f"\nTotal files: {len(all_members):,}")
    print(f"JSON files: {len(json_files):,}")
    
    # 첫 5개 샘플 읽기
    print("\n=== Sample Data (first 5) ===")
    samples = []
    for i, member in enumerate(json_files[:5]):
        f = tar.extractfile(member)
        if f:
            data = json.loads(f.read().decode('utf-8'))
            samples.append(data)
            
            print(f"\n[{i+1}] {member.name.split('/')[-1]}")
            print(f"  Category: {data['metadata']['category']}")
            print(f"  Source: {data['metadata']['source']}")
            print(f"  Sentences: {data['metadata']['num_sents']}")
            print(f"  Tokens: {data['metadata']['num_tokens']}")
            print(f"  Text: {data['texts'][0][:80]}...")

# 카테고리 분포
print("\n=== Category Distribution ===")
with tarfile.open(tar_path, 'r:gz') as tar:
    categories = {}
    for member in tar.getmembers():
        if member.name.endswith('.json'):
            parts = member.name.split('/')
            if len(parts) >= 2:
                cat = parts[1]
                categories[cat] = categories.get(cat, 0) + 1
    
    for cat, count in sorted(categories.items(), key=lambda x: x[1], reverse=True):
        print(f"  {cat}: {count:,} files")

# 통계
total_sents = sum(s['metadata']['num_sents'] for s in samples)
total_tokens = sum(s['metadata']['num_tokens'] for s in samples)
print(f"\n=== Statistics (5 samples) ===")
print(f"Total sentences: {total_sents}")
print(f"Total tokens: {total_tokens}")
print(f"Avg tokens/sample: {total_tokens / len(samples):.1f}")

## 3. 중국어 데이터셋 탐색

In [ ]:
# 중국어 데이터셋
zh_dataset_name = "LDCC/Final_Verification_Data_Labeling_Data_03.Multilingual_Pre-Learning_Data_02.Chinese"

print("[Chinese Dataset]")
print("Downloading TAR file...")

# TAR 파일 다운로드
tar_path = hf_hub_download(
    repo_id=zh_dataset_name,
    filename="Final_Verification_Data_Labeling_Data_03.Multilingual_Pre-Learning_Data_02.Chinese.tar.gz",
    repo_type="dataset"
)
print(f"✅ Downloaded: {tar_path}")

# TAR 구조 확인
with tarfile.open(tar_path, 'r:gz') as tar:
    all_members = tar.getmembers()
    json_files = [m for m in all_members if m.name.endswith('.json')]
    
    print(f"\nTotal files: {len(all_members):,}")
    print(f"JSON files: {len(json_files):,}")
    
    # 첫 5개 샘플 읽기
    print("\n=== Sample Data (first 5) ===")
    samples = []
    for i, member in enumerate(json_files[:5]):
        f = tar.extractfile(member)
        if f:
            data = json.loads(f.read().decode('utf-8'))
            samples.append(data)
            
            print(f"\n[{i+1}] {member.name.split('/')[-1]}")
            print(f"  Category: {data['metadata']['category']}")
            print(f"  Source: {data['metadata']['source']}")
            print(f"  Sentences: {data['metadata']['num_sents']}")
            print(f"  Tokens: {data['metadata']['num_tokens']}")
            print(f"  Text: {data['texts'][0][:80]}...")

# 카테고리 분포
print("\n=== Category Distribution ===")
with tarfile.open(tar_path, 'r:gz') as tar:
    categories = {}
    for member in tar.getmembers():
        if member.name.endswith('.json'):
            parts = member.name.split('/')
            if len(parts) >= 2:
                cat = parts[1]
                categories[cat] = categories.get(cat, 0) + 1
    
    for cat, count in sorted(categories.items(), key=lambda x: x[1], reverse=True):
        print(f"  {cat}: {count:,} files")

# 통계
total_sents = sum(s['metadata']['num_sents'] for s in samples)
total_tokens = sum(s['metadata']['num_tokens'] for s in samples)
print(f"\n=== Statistics (5 samples) ===")
print(f"Total sentences: {total_sents}")
print(f"Total tokens: {total_tokens}")
print(f"Avg tokens/sample: {total_tokens / len(samples):.1f}")

## 4. 데이터셋 요약 및 토크나이저 학습 계획

In [ ]:
print("=" * 70)
print("LDCC Dataset Summary")
print("=" * 70)

print("\n[Available Data]")
print(f"Japanese: 1,605,024 samples (~1,815 tokens/sample)")
print(f"Chinese:    899,327 samples (~2,535 tokens/sample)")
print(f"Korean:   unlimited (KORMo-Team/korean-web-collection)")
print(f"English:  unlimited (KORMo-Team/dclm-baseline-filtered)")

print("\n[Tokenizer Training Options for 5M samples]")
print("\nOption 1: Korean-centric (KO 50%, EN 30%, JA 10%, ZH 10%)")
print("  Korean:   2,500,000 samples (50%)")
print("  English:  1,500,000 samples (30%)")
print("  Japanese:   500,000 samples (10%) ✅")
print("  Chinese:    500,000 samples (10%) ✅")
print("  Vocab size: 100k")

print("\nOption 2: Data-constrained (KO 40%, EN 30%, JA 18%, ZH 12%)")
print("  Korean:   2,000,000 samples (40%)")
print("  English:  1,500,000 samples (30%)")
print("  Japanese:   900,000 samples (18%) ✅")
print("  Chinese:    600,000 samples (12%) ✅")
print("  Vocab size: 100k")

print("\n[Current Decision]")
print("✅ Using 7M Korean+English tokenizer (vocab: 80k)")
print("   - Korean: 70%, English: 30%")
print("   - Efficiency: 2.83 chars/token (KO), 6.20 chars/token (EN)")
print("   - Overall: +44.8% vs GPT-2")

print("\n[Future Work]")
print("- Multilingual tokenizer can be trained later if needed")
print("- Custom data loader required for LDCC datasets (TAR+JSON format)")
print("=" * 70)

## 5. 커스텀 데이터 로더 (참고용)

향후 다국어 토크나이저 학습 시 사용할 수 있는 커스텀 로더 예제

In [ ]:
def load_ldcc_texts(dataset_name, tar_filename, max_samples=None):
    """LDCC TAR 파일에서 텍스트 추출
    
    Args:
        dataset_name: HF dataset repo ID
        tar_filename: TAR 파일명
        max_samples: 최대 샘플 수 (None이면 전체)
        
    Yields:
        텍스트 문자열
    """
    import tarfile
    import json
    from huggingface_hub import hf_hub_download
    
    # TAR 다운로드
    tar_path = hf_hub_download(
        repo_id=dataset_name,
        filename=tar_filename,
        repo_type="dataset"
    )
    
    # JSON 파일 읽기
    count = 0
    with tarfile.open(tar_path, 'r:gz') as tar:
        for member in tar.getmembers():
            if member.name.endswith('.json'):
                if max_samples and count >= max_samples:
                    break
                    
                f = tar.extractfile(member)
                if f:
                    data = json.loads(f.read().decode('utf-8'))
                    # texts 배열의 모든 문장 합치기
                    text = ' '.join(data['texts'])
                    yield text
                    count += 1

# 사용 예제
print("[Usage Example]")
print("\n# 일본어 데이터 로드")
print("ja_texts = load_ldcc_texts(")
print("    dataset_name='LDCC/...Japanese',")
print("    tar_filename='...Japanese.tar.gz',")
print("    max_samples=500000  # 50만 샘플")
print(")")
print("\n# 중국어 데이터 로드")
print("zh_texts = load_ldcc_texts(")
print("    dataset_name='LDCC/...Chinese',")
print("    tar_filename='...Chinese.tar.gz',")
print("    max_samples=500000")
print(")")